# Buzzlytics — YOLOv8 Fine-Tuning (3-class: bee / pollen_bee / varroa_bee)

Trains **YOLOv8n** on a unified dataset built from two public research datasets:

- **VnPollenBee** (HUST ComVis) — hive-entrance detection, `nonpollenbee`→`bee`, `pollenbee`→`pollen_bee`.
- **VarroaDataset** (TU Wien, Zenodo 4085044) — varroa crops, whole-crop box → `varroa_bee` (infected) / `bee` (healthy).

`wasp` was dropped — neither dataset contains it (model is now 3-class).

## How it works
The notebook **clones the repo**, runs `datasets/prepare_dataset.py` (downloads + converts both
datasets, ~1.5 GB), then **caches the prepared dataset to your Google Drive** so future runs skip
the slow rebuild. No manual zipping/uploading required.

## Run
1. Runtime → Change runtime type → **GPU** (free T4 is enough).
2. Runtime → **Run all**. Approve the Google Drive mount when prompted.

**Class balance note:** the set is imbalanced — varroa/bee common, **pollen sparse (~1.8k instances)**.
Expect lower mAP on `pollen_bee`; document as a known limitation in the report. The VarroaDataset crops
are tightly-cropped single bees (domain gap vs hive-entrance scenes) — a deliberate tradeoff of the
whole-crop-box approach for the varroa class.

In [ ]:
# === 0. Configuration ===
REPO_URL = "https://github.com/okayxsh/Buzzlytics---CSCI435.git"
REPO_DIR = "/content/Buzzlytics---CSCI435"

# Google Drive is OPTIONAL — used only to cache the prepared dataset so you
# don't re-download ~1.5 GB every session. Set USE_DRIVE=False to skip Drive
# entirely (dataset rebuilds each run, ~10 min download).
USE_DRIVE = True
DRIVE_ZIP = "/content/drive/MyDrive/bee_dataset_3class.zip"  # cache path (if USE_DRIVE)
FORCE_REBUILD = False  # rebuild even if a cache exists

# Varroa crops are redundant single-bee shots (13.5k of them) and would swamp
# the ~2k real hive-entrance images. Cap them (class-balanced). None = all 13.5k.
VARROA_LIMIT = 3000

EPOCHS = 100
IMGSZ  = 640
BATCH  = 64            # A100 40GB has plenty of room; nano is tiny
WORKERS = 8            # feed the GPU faster (CPU decode is the bottleneck)
MODEL  = "yolov8n.pt"  # nano: justified by the 10-FPS requirement

In [ ]:
# === 1. Confirm GPU ===
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (switch Runtime -> GPU!)")

In [ ]:
# === 2. Install dependencies ===
!pip -q install "ultralytics==8.4.71" gdown pyyaml

In [ ]:
# === 3. (Optional) Mount Google Drive for the dataset cache ===
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print("USE_DRIVE=False -> skipping Drive; dataset will be built fresh each run.")

In [ ]:
# === 4. Clone (or update) the Buzzlytics repo — source of truth for prepare_dataset.py ===
import os
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("repo exists -> pulling latest")
    !cd {REPO_DIR} && git pull --ff-only
%cd {REPO_DIR}

In [ ]:
# === 5. Build the dataset (or load it from the Drive cache) ===
import os, zipfile, shutil, subprocess

DATA_DIR = f"{REPO_DIR}/datasets/data"
DATASETS_DIR = f"{REPO_DIR}/datasets"

have_cache = USE_DRIVE and os.path.isfile(DRIVE_ZIP) and not FORCE_REBUILD
if have_cache:
    print("Found cached dataset in Drive:", DRIVE_ZIP, "-> extracting (skipping rebuild)")
    shutil.rmtree(DATA_DIR, ignore_errors=True)
    with zipfile.ZipFile(DRIVE_ZIP) as z:
        z.extractall(DATASETS_DIR)
else:
    print("Building 3-class dataset from VnPollenBee + VarroaDataset (downloads ~1.5 GB)...")
    shutil.rmtree(DATA_DIR, ignore_errors=True)

    # VnPollenBee's Drive folder has >50 files per subfolder, so gdown's
    # scraper cannot list it. Pull the LabelMe JSONs via the real Drive API
    # (Colab auth, paginated). The JSONs carry embedded base64 imageData.
    import io
    from concurrent.futures import ThreadPoolExecutor
    from google.colab import auth
    from googleapiclient.discovery import build as gbuild
    from googleapiclient.http import MediaIoBaseDownload
    auth.authenticate_user()
    VPB_FOLDER_ID = "1fdEcu7CNmEkVAamu9wh_Ppw_-uW3VNY1"
    VPB_DIR = "/content/vnpollenbee"

    def _api():
        # one client per thread (googleapiclient http is not thread-safe)
        return gbuild("drive", "v3", cache_discovery=False)

    def _list_children(api, fid):
        items, tok = [], None
        while True:
            r = api.files().list(
                q=f"'{fid}' in parents and trashed=false",
                fields="nextPageToken, files(id, name, mimeType)",
                pageSize=1000, pageToken=tok,
                supportsAllDrives=True, includeItemsFromAllDrives=True,
            ).execute()
            items += r["files"]; tok = r.get("nextPageToken")
            if not tok: break
        return items

    # Phase 1: recursively enumerate every JSON -> (id, dest_path).
    def _walk(api, fid, dest):
        os.makedirs(dest, exist_ok=True)
        jobs = []
        for f in _list_children(api, fid):
            if f["mimeType"] == "application/vnd.google-apps.folder":
                jobs += _walk(api, f["id"], os.path.join(dest, f["name"]))
            elif f["name"].lower().endswith(".json"):
                jobs.append((f["id"], os.path.join(dest, f["name"])))
        return jobs

    # Phase 2: download all JSONs in parallel (~16 threads).
    import threading
    _tl = threading.local()
    def _get(job):
        if not hasattr(_tl, "api"): _tl.api = _api()
        fid, path = job
        req = _tl.api.files().get_media(fileId=fid)
        dl = MediaIoBaseDownload(io.FileIO(path, "wb"), req)
        done = False
        while not done:
            _, done = dl.next_chunk()

    shutil.rmtree(VPB_DIR, ignore_errors=True)
    print("Enumerating VnPollenBee Drive folder via Drive API...")
    jobs = _walk(_api(), VPB_FOLDER_ID, VPB_DIR)
    print(f"  {len(jobs)} JSON files -> downloading in parallel...")
    with ThreadPoolExecutor(max_workers=16) as ex:
        list(ex.map(_get, jobs))
    njson = len(jobs)
    print(f"  pulled {njson} JSON files")

    cmd = ["python", "datasets/prepare_dataset.py", "--vnpollenbee-dir", VPB_DIR]
    if VARROA_LIMIT is not None:
        cmd += ["--varroa-limit", str(VARROA_LIMIT)]
    subprocess.check_call(cmd)
    if USE_DRIVE:
        print("Caching prepared dataset to Drive:", DRIVE_ZIP)
        shutil.make_archive(DRIVE_ZIP[:-4], "zip", root_dir=DATASETS_DIR, base_dir="data")
print("DATA_DIR:", DATA_DIR)

In [ ]:
# === 6. Sanity check: class distribution (0=bee 1=pollen 2=varroa) ===
from collections import Counter
from pathlib import Path
c = Counter()
for txt in Path(DATA_DIR).rglob("labels/*.txt"):
    for line in txt.read_text().splitlines():
        if line.strip():
            c[int(line.split()[0])] += 1
print("instances per class:", dict(sorted(c.items())))
for s in ["train", "valid", "test"]:
    d = Path(DATA_DIR) / s / "images"
    print(f"{s}: {len(list(d.glob('*'))) if d.is_dir() else 0} images")

In [ ]:
# === 7. Write the dataset yaml (absolute path to the extracted data) ===
import yaml
data_cfg = {
    "path": DATA_DIR,
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": 3,
    "names": {0: "bee", 1: "pollen_bee", 2: "varroa_bee"},
}
with open("/content/bee_dataset.yaml", "w") as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False)
print(open("/content/bee_dataset.yaml").read())

In [ ]:
# === 8. Fine-tune YOLOv8n ===
from ultralytics import YOLO
model = YOLO(MODEL)
results = model.train(
    data="/content/bee_dataset.yaml",
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    name="bee_detector",
    patience=25,          # early stop if no val improvement
    seed=42,
    workers=WORKERS,
    # augmentations (report these in the write-up):
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    fliplr=0.5, mosaic=1.0, degrees=5.0,
)

In [ ]:
# === 9. Validate + per-class metrics (for the report's results table) ===
best = "runs/detect/bee_detector/weights/best.pt"
metrics = YOLO(best).val(data="/content/bee_dataset.yaml", split="test")
print("\nmAP@50:     ", round(float(metrics.box.map50), 4))
print("mAP@50-95:  ", round(float(metrics.box.map), 4))
names = {0: "bee", 1: "pollen_bee", 2: "varroa_bee"}
print("\nPer-class mAP@50:")
for i, ap in zip(metrics.box.ap_class_index, metrics.box.ap50):
    print(f"  {names.get(int(i), i):<12} {float(ap):.4f}")

In [ ]:
# === 10. Download best.pt ===
# Place the downloaded file at  cv_pipeline/weights/best.pt  in your local repo.
from google.colab import files
files.download("runs/detect/bee_detector/weights/best.pt")

# Also save the training plots (confusion matrix, PR curves) for the report:
!ls runs/detect/bee_detector/*.png